# Distributed Edge-AI for Modular Multilevel Converters

This notebook is an interactive walkthrough of a **10-cell MMC** in which every
cell is an independent **edge node** that runs a local fault-detection neural
network, with all nodes synchronised by **IEEE 1588 PTP**.

It demonstrates four things:

1. **Sub-microsecond synchronisation** between distributed nodes (PTP).
2. **Local AI inference** on each cell for fault detection.
3. A **distributed-vs-centralized** latency comparison.
4. **FPGA resource estimates** for quantised deployment (32/16/8/4-bit).

Use the interactive explorer near the bottom to sweep the number of cells,
model size and sync period and watch accuracy / latency update live.


## 1. Setup

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Resolve the project root whether the notebook runs from notebooks/ or root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import random
import numpy as np
import matplotlib.pyplot as plt
import torch

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED); random.seed(SEED)

from distributed_control import DistributedController
from benchmarks import quantization_sweep, estimate_fpga_resources, main as run_pipeline
from synchronization import PTPNetwork, PTPConfig

print("Imports OK -- project root:", ROOT)


## 2. Run the full pipeline

This trains the detector, runs the closed-loop distributed simulation, the
quantisation sweep and the FPGA analysis, and writes four figures plus
`summary.txt` into `results/`. It completes in a couple of seconds.


In [ ]:
outputs = run_pipeline(num_cells=10, num_steps=1000, model_size="small", headline_bits=8)
res = outputs["results"]
cmp = outputs["comparison"]
quant = outputs["quant"]


## 3. Headline results

In [ ]:
print(f"Cells / edge nodes        : {res.num_cells}")
print(f"Fault-detection accuracy  : {res.accuracy_pct:.2f} %")
print(f"Detection latency         : {res.detection_latency_ms:.2f} ms after onset")
print(f"Mean / max PTP sync error : {res.sync_error_mean_ns:.0f} / {res.sync_error_max_ns:.0f} ns")
print(f"Distributed AI speedup    : {cmp['speedup']:.2f} x")
print(f"Edge inference / call     : {cmp['edge_inference_ms']*1e3:.2f} us")


## 4. PTP synchronisation convergence

Each slave clock starts tens of microseconds off and is pulled to within a few
hundred nanoseconds of the grandmaster within a few sync intervals. The black
curve is the worst-case node; the red line is the 1 us target.


In [ ]:
net = PTPNetwork(num_nodes=10, config=PTPConfig(sync_interval_s=10e-3), seed=SEED)
r = net.run(num_cycles=200)
hist = np.abs(r["sync_error_history"]) / 1e-9   # ns
t_ms = r["time"] * 1e3

fig, ax = plt.subplots(figsize=(10, 5))
for node in range(1, hist.shape[1]):
    ax.plot(t_ms, hist[:, node], alpha=0.5, lw=1)
ax.plot(t_ms, hist[:, 1:].max(axis=1), "k", lw=2.2, label="worst-case node")
ax.axhline(1000, color="crimson", ls="--", label="1 us target")
ax.set_yscale("log"); ax.set_xlabel("Time (ms)"); ax.set_ylabel("Sync error (ns)")
ax.set_title("PTP convergence"); ax.legend(); plt.show()

print(f"steady mean = {r['mean_error_ns']:.0f} ns,  convergence = {r['convergence_time_ms']:.0f} ms")


## 5. Fault detection timeline

A thermal fault is injected on one cell at step 500. Its temperature ramps past
the 70 degC limit and the local detector flags it almost immediately.


In [ ]:
temp = res.fault_cell_temperature
steps = np.arange(len(temp))
t_ms = steps * res.dt * 1e3

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_ms, temp, color="#e76f51", lw=2, label=f"cell {res.fault_cell} temperature")
ax.axhline(70, color="black", ls="--", lw=1.3, label="70 degC fault limit")
ax.axvline(500 * res.dt * 1e3, color="grey", ls=":", lw=1.5, label="fault onset")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Temperature (degC)")
ax.set_title("Faulted-cell temperature"); ax.legend(); plt.show()


## 6. Interactive explorer

Move the sliders to change the number of cells (5-20), the edge model size,
the PTP sync period (1-100 ms) and the quantisation bit-width. Accuracy, sync
error, speedup and the FPGA footprint update on each change.

> Requires `ipywidgets` (`pip install ipywidgets`). Without it, the cell falls
> back to a small static sweep so it still runs end-to-end.


In [ ]:
def explore(num_cells=10, model_size="small", sync_period_ms=10, bits=8):
    ctrl = DistributedController(
        num_cells=num_cells, model_size=model_size,
        sync_interval_ms=sync_period_ms, bits=bits, seed=SEED,
    )
    r = ctrl.run(num_steps=600)
    c = ctrl.latency_comparison(r)
    fpga = estimate_fpga_resources(ctrl.detector, bits)
    print(f"cells={num_cells}  model={model_size}  sync={sync_period_ms} ms  bits={bits}")
    print(f"  accuracy         : {r.accuracy_pct:6.2f} %")
    print(f"  mean sync error  : {r.sync_error_mean_ns:6.0f} ns")
    print(f"  distributed speedup: {c['speedup']:.2f} x")
    print(f"  FPGA LUTs/cell   : {fpga.luts:5d}   latency {fpga.latency_us:.3f} us")


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(
        explore,
        num_cells=widgets.IntSlider(min=5, max=20, step=1, value=10),
        model_size=widgets.Dropdown(options=["tiny", "small", "medium"], value="small"),
        sync_period_ms=widgets.IntSlider(min=1, max=100, step=1, value=10),
        bits=widgets.Dropdown(options=[32, 16, 8, 4], value=8),
    )
except ImportError:
    print("ipywidgets not installed -- static sweep over cell counts:\n")
    for nc in (5, 10, 20):
        explore(num_cells=nc)
        print()


## 7. From simulation to FPGA

On real hardware each MMC cell carries a small SoC/FPGA (e.g. a Xilinx
**Zynq-7000**). The pieces in this repo map directly onto a deployment:

| Simulation component | FPGA implementation |
|----------------------|---------------------|
| `FaultDetector` (MLP) | Quantised fixed-point MAC datapath in fabric |
| `quantize_model` (8/4-bit) | Reduced-precision weights -> fewer LUTs/DSPs |
| `PTPNetwork` servo | Hardware PTP timestamping unit + clock servo |
| `DistributedController` | Per-cell real-time control loop |

The resource estimates show why quantisation matters: dropping from 32-bit
float to 8-bit fixed point cuts the LUT footprint by ~4x and removes all DSP
usage, while accuracy stays essentially unchanged -- exactly the trade that
makes per-cell edge inference practical.
